In [ ]:
n = 5000

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

#G = nx.read_adjlist(f'adjlists/ba_{n}.adjlist')
G = nx.read_adjlist(f'pscan_data/adjlists/lfr_{n}.adjlist')

# each node has a 'community' attribute
df = pd.read_csv(f"pscan_data/labels/lfr_{n}.labels.tsv", sep="\t")
num_communities = df["label"].nunique()
G.number_of_nodes(), G.number_of_edges(), num_communities

(5000, 37931, 285)

In [ ]:
gt_file = f"labels/lfr_{n}.labels.tsv"
gt_df = pd.read_csv(gt_file, sep="\t").rename(
    columns={"label": "gt_label"}
)
gt_df["node"] = gt_df["node"].astype(str)
gt_df["gt_label"] = gt_df["gt_label"].astype(int)
gt_df

,node,gt_label
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
159995,159995,3498
159996,159996,3034
159997,159997,2717
159998,159998,1839


In [ ]:
print(type(next(iter(G.nodes()))))
print(gt_df["node"].dtype)
print(list(G.nodes())[:5])
print(gt_df["node"].head().tolist())

<class 'str'>
object
['0', '3502', '5793', '24829', '25923']
['0', '1', '2', '3', '4']


In [ ]:
import networkx as nx

# gt_df has columns: node, gt_label
label_map = dict(zip(gt_df["node"], gt_df["gt_label"]))

# add gt_label as a node attribute
for i in G.nodes():
    G.nodes[i]["gt_label"] = int(label_map.get(i, -1))

# export for Gephi
nx.write_gexf(G, f"graph_gephi_{n}.gexf")

In [ ]:
import ast

# ----------------------------
# 2. Read cluster file
# ----------------------------
clusters_path = f"clusters/lfr_{n}.sim_clusters.csv"
clusters_df = pd.read_csv(clusters_path, header=None, names=["cluster_id", "nodes"])

# Convert string representation of lists into actual lists
clusters_df["nodes"] = clusters_df["nodes"].apply(ast.literal_eval)

# ----------------------------
# 3. Build node -> cluster mapping
# ----------------------------
node_to_cluster = {}

for _, row in clusters_df.iterrows():
    cluster_id = int(row["cluster_id"])
    for node in row["nodes"]:
        node_to_cluster[str(node)] = cluster_id   # graph nodes are strings

# ----------------------------
# 4. Attach cluster labels to graph nodes
# ----------------------------
missing = 0
for i in G.nodes():
    if i in node_to_cluster:
        G.nodes[i]["cluster"] = int(node_to_cluster[i])
    else:
        G.nodes[i]["cluster"] = -1
        missing += 1

print("Missing labels:", missing)

# ----------------------------
# 5. Save in Gephi format
# ----------------------------
nx.write_gexf(G, f"graph_with_clusters_{n}.gexf")

print(f"Saved graph_with_clusters_{n}.gexf")

Missing labels: 26464
Saved graph_with_clusters_160000.gexf
